### After Training the model, test it on a few select images

In [1]:
from PIL import Image, ImageOps
import face_recognition
import os

# crop single image

def crop_test_image(dirname, filename, output_dirname):
    image = face_recognition.load_image_file(os.path.join(dirname, filename))
    face_locations = face_recognition.face_locations(image, model="hog")

    if len(face_locations) > 0: # face found in image

        # Get Cropped Image
        height, width = image.shape[:2]
        top, right, bottom, left = face_locations[0]
        
        box_h = bottom - top
        box_w = right - left
        pad_h = int(box_h * 0.33) # add padding to get hair, neck, etc.
        pad_w = int(box_w * 0.33)

        top = max(0, top-pad_h)
        bottom = min(height, bottom+pad_h)
        left = max(0, left-pad_w)
        right = min(width, right+pad_w)

        face_image = image[top:bottom, left:right]
        pil_image = Image.fromarray(face_image)

        # add padding
        processed_image = ImageOps.pad(
            pil_image,
            (224,224),
            Image.Resampling.LANCZOS,
            (0,0,0) # black padding
        )

        processed_image.save(os.path.join(output_dirname, f"cropped_{filename}"))


# crop all test images

test_image_dir = "./datasets/faces/test_images"
test_output_dir = "./datasets/faces/test_images_cropped"
os.makedirs(test_output_dir, exist_ok=True)

test_images = os.listdir(test_image_dir)
for test_image in test_images:
    crop_test_image(test_image_dir, test_image, test_output_dir)

/home/mklema/ml_group_project/.venv/lib/python3.11/site-packages/face_recognition_models/__init__.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


In [2]:
import numpy as np
from PIL import Image
from tensorflow import keras
import os

# load model

model_path = "cnn_models/cnn_model_best.keras"
test_model = keras.models.load_model(model_path, compile=False)

# Test on each image

cropped_images = os.listdir(test_output_dir)
for ci in cropped_images:
    img = Image.open(os.path.join(test_output_dir, ci))
    img_array = np.array(img, dtype=np.float32)
    img_batch = np.expand_dims(img_array, axis=0)

    pred = test_model.predict(img_batch, verbose=0)
    predicted_age = float(pred[0][0])
    print(f"Predicted age of {ci}: {predicted_age:.2f}")

I0000 00:00:1773682274.015266 2144633 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1773682274.084783 2144633 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773682277.591584 2144633 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
E0000 00:00:1773682279.564421 2144633 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Predicted age of cropped_bella.jpg: 33.95
Predicted age of cropped_old.jpg: 98.12
Predicted age of cropped_max_image.jpg: 30.97
Predicted age of cropped_lisa.jpg: 51.12
Predicted age of cropped_olivia.jpg: 18.52


In [3]:
cropped_images = os.listdir(test_output_dir)
for ci in cropped_images:
    os.remove(os.path.join(test_output_dir, ci))
os.removedirs(test_output_dir)